In [12]:
import re

In [2]:
import os, json, time, glob
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
META_DIR = RAW / "meta"
META_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")

token = os.getenv("META_ACCESS_TOKEN") or ""
reports = sorted(glob.glob(str(RAW / "*advertisers*.csv")))

checks = {
    "Project root found": (ROOT / "pyproject.toml").exists(),
    ".env file exists": (ROOT / ".env").exists(),
    "Meta token loaded": bool(token),
    "Token looks valid (starts EAA, no spaces)": token.startswith("EAA") and " " not in token,
    "Ad Library Report CSV in data/raw": len(reports) > 0,
    "data/raw/meta folder ready": META_DIR.exists(),
}
for name, ok in checks.items():
    print(f"{'OK  ' if ok else 'FAIL'}  {name}")

OK    Project root found
OK    .env file exists
OK    Meta token loaded
OK    Token looks valid (starts EAA, no spaces)
OK    Ad Library Report CSV in data/raw
OK    data/raw/meta folder ready


In [3]:
r = requests.get(
    "https://graph.facebook.com/v27.0/ads_archive",
    params={
        "access_token": token,
        "ad_type": "POLITICAL_AND_ISSUE_ADS",
        "ad_reached_countries": '["US"]',
        "search_terms": "''",
        "fields": "page_name",
        "limit": 1,
    },
    timeout=30,
)
body = r.json()
print(r.status_code, body.get("data", body.get("error", {}).get("message")))

200 [{'page_name': 'News and Promotions', 'id': '2466562520504706'}]


In [4]:
report = pd.read_csv(reports[0])
print(Path(reports[0]).name, "|", report.shape)
print(report.columns.tolist())
report.head()

FacebookAdLibraryReport_2026-11-16_US_last_90_days_advertisers.csv | (32703, 5)
['Page ID', 'Page name', 'Disclaimer', 'Amount spent (USD)', 'Number of ads in Library']


,Page ID,Page name,Disclaimer,Amount spent (USD),Number of ads in Library
0,108107432370909,Kalshi - Trade the Headlines,KALSHI INC.,18874402,479
1,1386969781166493,Common Sense Voters Coalition,MAGA INC,16872992,8238
2,212559768612959,WinSenate,WINSENATE,8665493,542
3,1370758889449189,Lower Costs Now,LOWER COSTS NOW,8028122,9832
4,318975057964832,America PAC,America PAC,7086189,1460


In [5]:
def find_col(df, *words):
    for c in df.columns:
        if all(w in c.lower() for w in words):
            return c
    raise KeyError(f"No column containing {words}: {df.columns.tolist()}")

col_id = find_col(report, "page", "id")
col_name = find_col(report, "page", "name")
col_spend = find_col(report, "amount", "spent")
print("Using columns:", col_id, "|", col_name, "|", col_spend)

rep = report.rename(columns={col_id: "page_id", col_name: "page_name", col_spend: "spend_usd"})

# small amounts appear as text like "≤100": strip symbols, convert to numbers
rep["spend_usd"] = pd.to_numeric(
    rep["spend_usd"].astype(str).str.replace(r"[^\d.]", "", regex=True), errors="coerce"
).fillna(0)

# one page can appear on several rows (one per "paid for by" disclaimer), so total by page
pages = (rep.groupby(["page_id", "page_name"], as_index=False)["spend_usd"].sum()
         .sort_values("spend_usd", ascending=False))
pages["cum_share"] = pages["spend_usd"].cumsum() / pages["spend_usd"].sum()

top_pages = pages.head(200).copy()
top_pages.to_csv(INTERIM / "meta_top200_pages.csv", index=False)

for n in [50, 100, 200]:
    print(f"Top {n:>3} pages: {pages['cum_share'].iloc[n - 1]:.0%} of reported spend")
top_pages.head(20)

Using columns: Page ID | Page name | Amount spent (USD)
Top  50 pages: 36% of reported spend
Top 100 pages: 46% of reported spend
Top 200 pages: 57% of reported spend


,page_id,page_name,spend_usd,cum_share
6489,108107432370909,Kalshi - Trade the Headlines,18939813,0.042377
28937,1386969781166493,Common Sense Voters Coalition,16872992,0.080129
11197,212559768612959,WinSenate,8665493,0.099517
28658,1370758889449189,Lower Costs Now,8028122,0.117479
12978,318975057964832,America PAC,7086189,0.133334
30255,1626884974030414,James Talarico,6321491,0.147478
10331,174371409720577,Abdul El-Sayed,5806635,0.160470
23778,1163832573477744,"No on 40: Protect Funding for Schools, Healthc...",5084003,0.171845
589,12301006942,Democrats,4869412,0.182740
1567,92021268117,Working America,4662525,0.193172


In [6]:
API = "https://graph.facebook.com/v27.0/ads_archive"
FIELDS = ("id,page_id,page_name,bylines,ad_creative_bodies,ad_creative_link_titles,"
          "ad_delivery_start_time,ad_delivery_stop_time,spend,impressions,currency,"
          "publisher_platforms,ad_snapshot_url")

def fetch_page_ads(page_ids, start="2026-09-01", end="2026-11-03"):
    params = {
        "access_token": token,
        "ad_type": "POLITICAL_AND_ISSUE_ADS",
        "ad_reached_countries": '["US"]',
        "ad_active_status": "ALL",
        "ad_delivery_date_min": start,
        "ad_delivery_date_max": end,
        "search_page_ids": json.dumps([int(p) for p in page_ids]),
        "fields": FIELDS,
        "limit": 250,
    }
    rows, url = [], API
    while url:
        r = requests.get(url, params=params, timeout=60)
        if r.status_code != 200:
            msg = r.json().get("error", {}).get("message", "unknown error")
            raise RuntimeError(f"Meta API error {r.status_code}: {msg}")
        data = r.json()
        rows.extend(data.get("data", []))
        url = data.get("paging", {}).get("next")
        params = None        # the "next" URL already carries every parameter
        time.sleep(1)
    return rows

In [7]:
batch_ids = top_pages["page_id"].head(10).tolist()
ads_batch = fetch_page_ads(batch_ids)

with open(META_DIR / "batch_000.json", "w") as f:
    json.dump(ads_batch, f)

meta = pd.DataFrame(ads_batch)
meta["body"] = meta["ad_creative_bodies"].apply(
    lambda b: " ".join(b).strip() if isinstance(b, list) else "")

print("Ads returned:", len(meta))
print("Share with empty text:", round((meta["body"] == "").mean(), 3))
print()
print(meta.groupby("page_name").size().sort_values(ascending=False))
meta[["page_name", "ad_delivery_start_time", "spend", "body"]].head(10)

Ads returned: 36945
Share with empty text: 0.49

page_name
Lower Costs Now                                                      9831
Common Sense Voters Coalition                                        8236
Working America                                                      7374
Abdul El-Sayed                                                       4693
Democrats                                                            3102
America PAC                                                          1404
No on 40: Protect Funding for Schools, Healthcare & Public Safety     985
WinSenate                                                             534
Kalshi - Trade the Headlines                                          496
James Talarico                                                        290
dtype: int64


,page_name,ad_delivery_start_time,spend,body
0,Kalshi - Trade the Headlines,2026-11-03,"{'lower_bound': '200', 'upper_bound': '299'}",This ad ran without a required disclaimer.
1,Kalshi - Trade the Headlines,2026-11-03,"{'lower_bound': '100', 'upper_bound': '199'}",This ad ran without a required disclaimer.
2,Kalshi - Trade the Headlines,2026-11-03,"{'lower_bound': '2500', 'upper_bound': '2999'}",This ad ran without a required disclaimer.
3,Kalshi - Trade the Headlines,2026-11-03,"{'lower_bound': '2500', 'upper_bound': '2999'}",This ad ran without a required disclaimer.
4,America PAC,2026-10-24,"{'lower_bound': '0', 'upper_bound': '99'}",Broken windows. Burning cities. Chaos in our s...
5,America PAC,2026-10-24,"{'lower_bound': '0', 'upper_bound': '99'}",The alarm clock says “go vote.” So do the stic...
6,America PAC,2026-10-24,"{'lower_bound': '0', 'upper_bound': '99'}",The alarm clock says “go vote.” So do the stic...
7,America PAC,2026-10-25,"{'lower_bound': '0', 'upper_bound': '99'}",Broken windows. Burning cities. Chaos in our s...
8,America PAC,2026-10-24,"{'lower_bound': '0', 'upper_bound': '99'}",The alarm clock says “go vote.” So do the stic...
9,America PAC,2026-10-24,"{'lower_bound': '0', 'upper_bound': '99'}",Broken windows. Burning cities. Chaos in our s...


In [8]:
STATES = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
          "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
          "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan",
          "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
          "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
          "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
          "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia",
          "Wisconsin", "Wyoming"]

offices = (r"(?:u\.?s\.? )?(?:senate|senator|congress|governor|house|attorney general|"
           r"lieutenant governor|lt\.? governor|secretary of state|treasurer|state senate|"
           r"state house|assembly|mayor)")
state_re = "|".join(s.lower() for s in STATES)

CANDIDATE_RE = (rf"\bfor {offices}\b|\bfor (?:{state_re})\b|\bfriends of\b|"
                rf"\bcommittee to elect\b|\belect [a-z]+")
PARTY_RE = (r"\b(?:democratic|republican) (?:party|national committee|state central|"
            r"county committee|senatorial|congressional)\b|\b(?:dnc|rnc|nrsc|dscc|dccc|nrcc)\b")

col_disc = find_col(report, "disclaimer")
rep["disclaimer"] = report[col_disc].fillna("").astype(str)
d = rep["disclaimer"].str.lower()

rep["page_type"] = "other"
rep.loc[d.str.contains(PARTY_RE, regex=True), "page_type"] = "party_committee"
rep.loc[d.str.contains(CANDIDATE_RE, regex=True), "page_type"] = "candidate"

by_page = (rep.groupby(["page_id", "page_name", "page_type"], as_index=False)
              .agg(spend_usd=("spend_usd", "sum"), disclaimer=("disclaimer", "first"))
              .sort_values("spend_usd", ascending=False))

print((by_page.groupby("page_type")["spend_usd"].sum() / by_page["spend_usd"].sum()).round(3))
by_page[by_page["page_type"] == "candidate"].head(30)

page_type
candidate          0.184
other              0.788
party_committee    0.028
Name: spend_usd, dtype: float64


,page_id,page_name,page_type,spend_usd,disclaimer
30439,1626884974030414,James Talarico,candidate,6162469,TALARICO FOR TEXAS
10378,174371409720577,Abdul El-Sayed,candidate,5806635,Abdul for U.S. Senate
15876,570953166908066,Vivek Ramaswamy,candidate,2742901,Vivek Ramaswamy and Rob McColley for Ohio
28056,1336293769761923,Jon Ossoff,candidate,2003426,JON OSSOFF FOR SENATE
7499,111025563615171,Tom Tiffany,candidate,1578957,Tiffany for Wisconsin
6799,108873101593827,Dr. Annie Andrews,candidate,1560632,DR. ANNIE ANDREWS FOR SENATE
23485,1138815389310118,Team Ossoff,candidate,1489510,JON OSSOFF FOR SENATE
6323,107553188401159,Brandon Gill,candidate,1482732,Brandon Gill for Texas
14300,410726399046796,Troy Jackson,candidate,1470143,Troy Jackson for Maine
415,6266829799,Sherrod Brown,candidate,1324436,Friends of Sherrod Brown


In [9]:
REMOVED = "this ad ran without a required disclaimer"

meta["body_clean"] = meta["body"].str.strip()
meta = meta[~meta["body_clean"].str.lower().str.startswith(REMOVED)]
meta = meta[meta["body_clean"] != ""]

# collapse near-identical copies of the same message
meta["body_key"] = meta["body_clean"].str.lower().str.replace(r"\s+", " ", regex=True)
unique_msgs = (meta.groupby(["page_name", "body_key"], as_index=False)
                   .agg(n_copies=("id", "count"), body=("body_clean", "first")))

print("Ads after cleaning:", len(meta), "| unique messages:", len(unique_msgs))

Ads after cleaning: 18809 | unique messages: 3026


In [10]:
shares = by_page.groupby("page_type")["spend_usd"].sum() / by_page["spend_usd"].sum()
print(shares.round(3))

# candidates the pattern may have missed: check this list by hand
by_page[by_page["page_type"] == "other"].head(30)[["page_name", "disclaimer", "spend_usd"]]

page_type
candidate          0.184
other              0.788
party_committee    0.028
Name: spend_usd, dtype: float64


,page_name,disclaimer,spend_usd
6523,Kalshi - Trade the Headlines,KALSHI INC.,18939813
29114,Common Sense Voters Coalition,MAGA INC,16872992
11248,WinSenate,WINSENATE,8665493
28834,Lower Costs Now,LOWER COSTS NOW,8028122
13040,America PAC,America PAC,7086189
23922,"No on 40: Protect Funding for Schools, Healthc...","No on Prop 40 - Doctors, Teachers, Small Busin...",5084003
1574,Working America,Working America,4662525
31003,Ground News,Ground News,4190436
14285,Turning Point PAC,TURNING POINT PAC INC.,3727609
22154,Yes on Proposition 42,"Yes on 42 – Protect Retirement & Life Savings,...",3260027


In [13]:
ORG_WORDS = {"america", "american", "americans", "action", "fund", "vote", "voters", "party",
             "committee", "coalition", "project", "alliance", "network", "majority", "leadership",
             "senate", "house", "working", "families", "people", "future", "united", "national",
             "state", "local", "election", "elections", "freedom", "liberty", "conservative",
             "progressive", "values", "point", "turning", "news", "media", "group", "foundation",
             "institute", "council", "union", "workers", "citizens", "first", "forward", "victory"}
ORG_WORDS |= {s.lower() for s in STATES} | {s.lower().rstrip("s") + "ns" for s in STATES}

def distinctive_words(s):
    return set(re.findall(r"[a-z]{4,}", str(s).lower())) - ORG_WORDS

other = by_page[(by_page["page_type"] == "other") & (by_page["spend_usd"] >= 10_000)].copy()
other["shared"] = other.apply(
    lambda r: distinctive_words(r["page_name"]) & distinctive_words(r["disclaimer"]), axis=1)
suspects = other[other["shared"].str.len() > 0]

print(len(suspects), "pages to review")
suspects[["page_name", "disclaimer", "spend_usd", "shared"]].head(60)

1998 pages to review


,page_name,disclaimer,spend_usd,shared
6523,Kalshi - Trade the Headlines,KALSHI INC.,18939813,{kalshi}
11248,WinSenate,WINSENATE,8665493,{winsenate}
28834,Lower Costs Now,LOWER COSTS NOW,8028122,"{costs, lower}"
23922,"No on 40: Protect Funding for Schools, Healthc...","No on Prop 40 - Doctors, Teachers, Small Busin...",5084003,"{public, safety}"
31003,Ground News,Ground News,4190436,{ground}
19745,One Nation,One Nation,3219458,{nation}
6300,WorkMoney,WorkMoney,2753908,{workmoney}
12209,Rep. Gina Hinojosa,Gina Hinojosa Campaign,2231846,"{hinojosa, gina}"
3960,Defend The Vote,DEFEND THE VOTE ACTION FUND,2194102,{defend}
1817,Dan Patrick,Texans For Dan Patrick,1885321,{patrick}


In [14]:
def looks_like_person(name):
    """Page names like 'Dan Patrick' or 'Rep. Gina Hinojosa': 2-4 capitalised words, no org words."""
    words = re.sub(r"^(rep|sen|gov|dr|mayor|judge)\.?\s+", "", str(name), flags=re.I).split()
    return (2 <= len(words) <= 4
            and all(w[:1].isupper() for w in words)
            and not distinctive_words(name) == set()
            and not (set(w.lower() for w in words) & ORG_WORDS))

other["person_name"] = other["page_name"].map(looks_like_person)
other["payer_style"] = other["disclaimer"].str.lower().str.contains(
    r"\bcampaign\b|\bfor\b|^[a-z]+ [a-z]+$", regex=True)

# 2 = person-like page name AND candidate-style payer; 1 = one of the two
other["score"] = other["person_name"].astype(int) + other["payer_style"].astype(int)
suspects = (other[(other["shared"].str.len() > 0) & (other["score"] >= 1)]
            .sort_values(["score", "spend_usd"], ascending=[False, False]))

total = suspects["spend_usd"].sum()
for n in [50, 100, 200]:
    print(f"Top {n:>3} suspects: {suspects['spend_usd'].head(n).sum() / total:.0%} of suspect spend")
print(len(suspects), "suspects after scoring")

suspects[["page_name", "disclaimer", "spend_usd", "score"]].head(50)

Top  50 suspects: 24% of suspect spend
Top 100 suspects: 26% of suspect spend
Top 200 suspects: 28% of suspect spend
1069 suspects after scoring


,page_name,disclaimer,spend_usd,score
19745,One Nation,One Nation,3219458,2
12209,Rep. Gina Hinojosa,Gina Hinojosa Campaign,2231846,2
1817,Dan Patrick,Texans For Dan Patrick,1885321,2
10921,Josh Shapiro,Josh Shapiro,1594802,2
5920,Mary Peltola,ALASKANS FOR MARY,1270613,2
18144,Political Identity,Political Identity,1071222,2
2243,Greg Abbott,Texans for Greg Abbott,1033051,2
15734,Dr. Amy Acton,Ohioans for Amy Acton,869623,2
1518,Kathy Hochul,Friends for Kathy Hochul,785340,2
13959,Hillsdale College,Hillsdale College,706816,2


In [15]:
REVIEWED = set(suspects.head(50)["page_id"])
remaining = suspects[~suspects["page_id"].isin(REVIEWED)]

In [17]:
export = (remaining[["page_id", "page_name", "disclaimer", "spend_usd", "score"]]
          .sort_values("spend_usd", ascending=False))
export.to_csv(INTERIM / "candidate_suspects_for_review.csv", index=False)
print(len(export), "rows | spend covered:",
      f"{export['spend_usd'].sum() / remaining['spend_usd'].sum():.0%} of remaining suspect spend")

1019 rows | spend covered: 100% of remaining suspect spend


In [19]:
MANUAL_CANDIDATE_PAYERS = [
    "Gina Hinojosa Campaign",
    "Texans For Dan Patrick",
    "Josh Shapiro",
    "ALASKANS FOR MARY",                                    # Mary Peltola
    "Texans for Greg Abbott",
    "Ohioans for Amy Acton",
    "Friends for Kathy Hochul",
    "Blaise Ingoglia for CFO",
    "Ken Paxton Campaign",
    "Don Huffines Campaign",
    "Minnesotans for Klobuchar",
    "Alaskans for Sullivan",                                # Dan Sullivan
    "James for NY 2026",                                    # Letitia James
    "John Kennedy for Us, Inc",
    "ALASKANS FOR NICK BEGICH",
    "KANSANS FOR MARSHALL",                                 # Roger Marshall
    "CITIZENS FOR GIANNOULIAS",
    "TEXANS FOR HENRY CUELLAR CONGRESSIONAL CAMPAIGN",
]

In [21]:

def norm_payer(s):
    s = str(s).lower()
    s = re.sub(r"[^\w\s]", " ", s)                 # drop punctuation
    s = re.sub(r"\b(inc|llc|the)\b", " ", s)       # drop legal suffixes
    return re.sub(r"\s+", " ", s).strip()

# the committees we confirmed by hand earlier
earlier = [
    "Gina Hinojosa Campaign", "Texans For Dan Patrick", "Josh Shapiro", "ALASKANS FOR MARY",
    "Texans for Greg Abbott", "Ohioans for Amy Acton", "Friends for Kathy Hochul",
    "Blaise Ingoglia for CFO", "Ken Paxton Campaign", "Don Huffines Campaign",
    "Minnesotans for Klobuchar", "Alaskans for Sullivan", "James for NY 2026",
    "John Kennedy for Us, Inc", "ALASKANS FOR NICK BEGICH", "KANSANS FOR MARSHALL",
    "CITIZENS FOR GIANNOULIAS", "TEXANS FOR HENRY CUELLAR CONGRESSIONAL CAMPAIGN",
]

# the reviewed file
rev = pd.read_csv(INTERIM / "candidate_suspects_reviewed.csv")
KEEP_CALLS = ["candidate", "party_committee"]        # add "judicial_candidate" if you include judges
confirmed = rev[rev["my_call"].isin(KEEP_CALLS)]

MANUAL_CANDIDATE_PAYERS = {norm_payer(p) for p in earlier} | {norm_payer(p) for p in confirmed["disclaimer"]}
pd.DataFrame({"payer": sorted(MANUAL_CANDIDATE_PAYERS)}).to_csv(INTERIM / "manual_candidate_payers.csv", index=False)

print(len(MANUAL_CANDIDATE_PAYERS), "manual payers")

266 manual payers


In [22]:
rev = pd.read_csv(INTERIM / "candidate_suspects_reviewed.csv")
KEEP_CALLS = ["candidate", "party_committee"]        # add "judicial_candidate" if you include judges
confirmed = rev[rev["my_call"].isin(KEEP_CALLS)]
MANUAL_CANDIDATE_PAYERS |= {norm_payer(p) for p in confirmed["disclaimer"]}
print(len(MANUAL_CANDIDATE_PAYERS), "manual payers")

266 manual payers


In [24]:
# normalised payer name for every row of the report
rep["payer"] = rep["disclaimer"].map(norm_payer)

# party leaders (same rule for both parties); leave empty for now if undecided
LEADER_PAYERS = [
    # "example leader for america",
]
LEADER_PAYERS = {norm_payer(p) for p in LEADER_PAYERS}

# keep: pattern-matched candidates and party committees, leaders, and your hand-confirmed list
rep["keep"] = (rep["page_type"].isin(["candidate", "party_committee"])
               | rep["payer"].isin(LEADER_PAYERS)
               | rep["payer"].isin(MANUAL_CANDIDATE_PAYERS))
sel = rep[rep["keep"]].copy()
KEEP_PAYERS = set(sel["payer"])

# affiliation: all pages paid for by the same committee
affil = (sel.groupby("payer")
            .agg(spend_usd=("spend_usd", "sum"),
                 n_pages=("page_id", "nunique"),
                 pages=("page_name", lambda x: sorted(set(x))),
                 page_ids=("page_id", lambda x: sorted(set(x))))
            .sort_values("spend_usd", ascending=False)
            .reset_index())
affil.to_csv(INTERIM / "meta_payer_affiliations.csv", index=False)

print("Committees (payers):", len(affil), "| pages:", sel["page_id"].nunique())
print("Committees running more than one page:", (affil["n_pages"] > 1).sum())
print(f"Spend covered: ${sel['spend_usd'].sum() / 1e6:.1f}M")
affil[affil["n_pages"] > 1].head(20)

Committees (payers): 4575 | pages: 4597
Committees running more than one page: 62
Spend covered: $116.1M


,payer,spend_usd,n_pages,pages,page_ids
0,talarico for texas,6812771,2,"[James Talarico, Team Talarico]","[717869048073637, 1626884974030414]"
3,jon ossoff for senate,3492936,2,"[Jon Ossoff, Team Ossoff]","[1138815389310118, 1336293769761923]"
7,alaskans for mary,1746503,3,"[Alaska Brief, Mary Peltola, Sullivan Watch]","[106612915352436, 1060677383795773, 1284051098..."
12,troy jackson for maine,1506341,2,"[Maine Event, Troy Jackson]","[410726399046796, 1420431821147003]"
15,jb for governor,1299664,3,"[Dangerous Darren Bailey, Illinois Daily, JB P...","[108437315253996, 320509205167290, 78375313847..."
16,hamilton for kansas,1235030,2,"[Adam Hamilton for Kansas, The Kansas Bulletin]","[1121557611034507, 1326865650510817]"
20,ohio democratic party,1071944,3,"[Dr. Amy Acton, Ohio Democratic Party, Vivek P...","[80892038525, 558219887366326, 1389325974253477]"
22,eric jones for congress,1031952,2,[Californians Against the Democratic Establish...,"[782932888227839, 1256264450911550]"
23,rob sand for iowa,1010070,2,"[Iowa Updates, Rob Sand for Iowa]","[505598353158784, 1227042123816013]"
24,dccc,1005863,3,"[California Values, Democrats, The American Ho...","[23790541544, 137160776146989, 171073822762250]"


In [25]:
def payer_words(p):
    return {w for w in p.split() if len(w) >= 4} - {"campaign", "committee", "friends", "citizens",
                                                     "elect", "team", "senate", "congress", "governor"}

page_style = sel[["page_id", "page_name", "payer"]].drop_duplicates()
page_style["own_name_page"] = page_style.apply(
    lambda r: bool(payer_words(r["payer"]) & set(r["page_name"].lower().split())), axis=1)
page_style.to_csv(INTERIM / "meta_page_style.csv", index=False)

print(page_style["own_name_page"].value_counts())
page_style[~page_style["own_name_page"]].head(30)

own_name_page
True     4466
False     197
Name: count, dtype: int64


,page_id,page_name,payer,own_name_page
8,12301006942,Democrats,dnc services corp dem nat l committee,False
26,123192635089,Republicans,republican national committee,False
60,20787991568,Senate Democrats,dscc,False
66,558219887366326,Dr. Amy Acton,ohio democratic party,False
70,23790541544,Democrats,dccc,False
104,108437315253996,Dangerous Darren Bailey,jb for governor,False
146,783753138471485,JB Pritzker,jb for governor,False
147,1375618656016062,Deb Haaland,deb for new mexico,False
158,1284051098116926,Sullivan Watch,alaskans for mary,False
299,107572881870115,Josh Turek,turek4iowa,False


In [26]:
GROUP_WORDS = r"\bpac\b|together|families|coalition|action|fund|alliance|too extreme|\bstop\b|\byes\b|\bno on\b|voters|citizens for (a|the)\b|project"
check = (sel.groupby("payer", as_index=False)["spend_usd"].sum()
            .sort_values("spend_usd", ascending=False))
check = check[check["payer"].str.contains(GROUP_WORDS, regex=True)]
print(len(check), "payers to review")
check.head(60)

40 payers to review


/tmp/ipykernel_89018/3010405334.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  check = check[check["payer"].str.contains(GROUP_WORDS, regex=True)]


,payer,spend_usd
1840,friends of byron donalds pac,409949
601,coalition for new york s future,346506
1242,cooper victory fund,319355
3675,ossoff talarico victory fund,314989
1709,for michigan action fund,294292
3911,republican leadership for arizona pac not auth...,249746
2552,friends of tina kotek pac id 4792,198222
4355,together for michigan,173436
3643,no on g save sunset dunes park sponsored by fr...,80568
4374,too extreme for idaho pac,66579


In [27]:
EXCLUDE_FRAGMENTS = [
    "coalition for new york", "for michigan action fund", "republican leadership for arizona",
    "together for michigan", "no on g save sunset", "too extreme for idaho",
    "valley families for melissa hurtado", "cap pac center for arizona", "friends of earth action",
    "yes on u friends of walnut", "alliance for michigan power", "friends of san bernardino community",
    "winning for louisiana", "all in for utah", "coalition for a clean and safe imperial",
    "veterans for idaho voters", "families for florida", "together we elect",
    "center for arizona policy action", "friends of santa rosa junior", "yes for washington elementary",
    "friends of bella vista voters", "friends of barstow community", "united for virginia s future",
    "families and friends of louisiana", "solidarity for michigan",
]

EXCLUDE_PAYERS = {p for p in KEEP_PAYERS if any(f in p for f in EXCLUDE_FRAGMENTS)}
print(len(EXCLUDE_PAYERS), "payers excluded:")
for p in sorted(EXCLUDE_PAYERS):
    print("  ", p)

KEEP_PAYERS = KEEP_PAYERS - EXCLUDE_PAYERS
pd.DataFrame({"payer": sorted(EXCLUDE_PAYERS)}).to_csv(INTERIM / "excluded_payers.csv", index=False)
print(len(KEEP_PAYERS), "payers kept")

26 payers excluded:
   all in for utah pac
   alliance for michigan power
   cap pac center for arizona policy political action committee not authorized by any candidate or candidate s committee 0 from out of state contributors
   center for arizona policy action
   coalition for a clean and safe imperial beach in support of leyba gonzalez for mayor 2026 sponsored by laborers international union of north america local 89
   coalition for new york s future
   families and friends of louisiana s incarcerated children fflic
   families for florida
   for michigan action fund
   friends of barstow community college yes on c 2026
   friends of bella vista voters
   friends of earth action
   friends of san bernardino community college yes on y fppc 1495565
   friends of santa rosa junior college foundation yes on measure ab
   no on g save sunset dunes park sponsored by friends of sunset dunes
   republican leadership for arizona pac not authorized by any candidate or candidate committee
  

In [28]:
class MetaAPIError(Exception):
    def __init__(self, code, msg):
        super().__init__(f"Meta API error {code}: {msg}")
        self.code = code

API = "https://graph.facebook.com/v27.0/ads_archive"
FIELDS = ("id,page_id,page_name,bylines,ad_creative_bodies,ad_creative_link_titles,"
          "ad_delivery_start_time,ad_delivery_stop_time,spend,impressions,currency,"
          "publisher_platforms,ad_snapshot_url")
RATE_LIMIT_CODES = {4, 17, 32, 613}
TOKEN_CODES = {190}

def fetch_page_ads(page_ids, start="2026-09-01", end="2026-11-03"):
    params = {
        "access_token": token,
        "ad_type": "POLITICAL_AND_ISSUE_ADS",
        "ad_reached_countries": '["US"]',
        "ad_active_status": "ALL",
        "ad_delivery_date_min": start,
        "ad_delivery_date_max": end,
        "search_page_ids": json.dumps([int(p) for p in page_ids]),
        "fields": FIELDS,
        "limit": 250,
    }
    rows, url = [], API
    while url:
        r = requests.get(url, params=params, timeout=90)
        if r.status_code != 200:
            err = r.json().get("error", {})
            raise MetaAPIError(err.get("code"), err.get("message", "unknown error"))
        data = r.json()
        rows.extend(data.get("data", []))
        url = data.get("paging", {}).get("next")
        params = None            # the "next" URL already carries every parameter
        time.sleep(1)
    return rows

def fetch_with_retry(ids, max_tries=5):
    for attempt in range(1, max_tries + 1):
        try:
            return fetch_page_ads(ids)
        except MetaAPIError as e:
            if e.code in TOKEN_CODES:
                raise RuntimeError("Token expired or invalid: generate a new one and update .env")
            if e.code in RATE_LIMIT_CODES and attempt < max_tries:
                wait = 300 * attempt
                print(f"  rate limited, waiting {wait // 60} min (attempt {attempt})")
                time.sleep(wait)
            else:
                raise
        except requests.exceptions.RequestException as e:
            print(f"  network problem ({type(e).__name__}), retrying in 60 s")
            time.sleep(60)
    raise RuntimeError("Gave up after repeated failures")

In [30]:
load_dotenv(ROOT / ".env", override=True)
token = os.getenv("META_ACCESS_TOKEN") or ""
print("Token loaded:", token.startswith("EAA"))

Token loaded: True


In [31]:
# only pages paid for by a payer we're keeping
pull_pages = sel[sel["payer"].isin(KEEP_PAYERS)]
page_ids = sorted(pull_pages["page_id"].astype(str).unique())
batches = [page_ids[i:i + 10] for i in range(0, len(page_ids), 10)]

# save the page list so the batch numbering can be reproduced
pd.Series(page_ids, name="page_id").to_csv(INTERIM / "meta_pull_page_ids.csv", index=False)
print(f"{len(page_ids)} pages in {len(batches)} batches")

start_time = time.time()
for i, ids in enumerate(batches):
    out = META_DIR / f"cand_batch_{i:04d}.json"
    if out.exists():
        continue                                  # already pulled, skip
    rows = fetch_with_retry(ids)
    out.write_text(json.dumps(rows))
    if i % 10 == 0:
        mins = (time.time() - start_time) / 60
        print(f"batch {i + 1}/{len(batches)} | {len(rows)} ads | {mins:.0f} min elapsed")

print("Done.")

4572 pages in 458 batches
batch 111/458 | 178 ads | 0 min elapsed
batch 121/458 | 39 ads | 1 min elapsed
batch 131/458 | 181 ads | 2 min elapsed
batch 141/458 | 99 ads | 2 min elapsed
batch 151/458 | 99 ads | 3 min elapsed
batch 161/458 | 5381 ads | 4 min elapsed
batch 171/458 | 175 ads | 6 min elapsed
batch 181/458 | 178 ads | 6 min elapsed
batch 191/458 | 301 ads | 8 min elapsed
batch 201/458 | 150 ads | 8 min elapsed
batch 211/458 | 99 ads | 9 min elapsed
batch 221/458 | 185 ads | 10 min elapsed
batch 231/458 | 106 ads | 11 min elapsed
batch 241/458 | 253 ads | 13 min elapsed
batch 251/458 | 192 ads | 14 min elapsed
batch 261/458 | 333 ads | 15 min elapsed
batch 271/458 | 709 ads | 16 min elapsed
batch 281/458 | 183 ads | 17 min elapsed
batch 291/458 | 287 ads | 18 min elapsed
batch 301/458 | 136 ads | 19 min elapsed
batch 311/458 | 178 ads | 20 min elapsed
batch 321/458 | 528 ads | 21 min elapsed
batch 331/458 | 336 ads | 22 min elapsed
batch 341/458 | 253 ads | 24 min elapsed
batc

In [32]:
all_rows = []
for f in sorted(META_DIR.glob("cand_batch_*.json")):
    all_rows.extend(json.loads(f.read_text()))
print("Raw ads pulled:", len(all_rows))

ads_m = pd.DataFrame(all_rows).drop_duplicates("id")

# keep only ads paid for by the committees we selected
ads_m["payer"] = ads_m["bylines"].fillna("").map(norm_payer)
ads_m = ads_m[ads_m["payer"].isin(KEEP_PAYERS)].copy()

# ad text: join the creative bodies, drop empty and removed ads
ads_m["body"] = ads_m["ad_creative_bodies"].apply(
    lambda b: " ".join(b).strip() if isinstance(b, list) else "")
REMOVED = "this ad ran without a required disclaimer"
ads_m = ads_m[(ads_m["body"] != "") & ~ads_m["body"].str.lower().str.startswith(REMOVED)]

# spend comes as a range: keep lower, upper and midpoint
def bound(s, key):
    if not isinstance(s, dict):
        return 0.0
    return float(s.get(key, s.get("lower_bound", 0)) or 0)
ads_m["spend_lo"] = ads_m["spend"].apply(lambda s: bound(s, "lower_bound"))
ads_m["spend_hi"] = ads_m["spend"].apply(lambda s: bound(s, "upper_bound"))
ads_m["spend_mid"] = (ads_m["spend_lo"] + ads_m["spend_hi"]) / 2

# page style from earlier (own-name page or not)
ads_m["page_id"] = ads_m["page_id"].astype(str)
ps = page_style.assign(page_id=page_style["page_id"].astype(str))[["page_id", "own_name_page"]].drop_duplicates("page_id")
ads_m = ads_m.merge(ps, on="page_id", how="left")

# collapse near-identical copies of the same message
ads_m["body_key"] = ads_m["body"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
msgs = (ads_m.groupby(["payer", "body_key"], as_index=False)
            .agg(body=("body", "first"),
                 n_copies=("id", "count"),
                 spend_mid=("spend_mid", "sum"),
                 first_seen=("ad_delivery_start_time", "min"),
                 own_name_page=("own_name_page", "max")))
msgs.insert(0, "msg_id", range(len(msgs)))

PROC = ROOT / "data" / "processed"
PROC.mkdir(exist_ok=True)
ads_m.drop(columns=["spend", "ad_creative_bodies"]).to_parquet(PROC / "meta_candidate_ads.parquet", index=False)
msgs.to_parquet(PROC / "meta_candidate_messages.parquet", index=False)

print("Ads kept:", len(ads_m), "| committees:", ads_m["payer"].nunique(), "| unique messages:", len(msgs))

Raw ads pulled: 139390
Ads kept: 129095 | committees: 4328 | unique messages: 40598
